In [ ]:
import pandas as pd
import numpy as np
import math


In [ ]:
df = pd.read_csv("wdbc.csv")

x = df.iloc[:, 2:]
y = df.iloc[:, 1:2]

rows, cols = x.shape

# Shuffle and split
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x.iloc[train_idx].values
x_test = x.iloc[test_idx].values
y_train = y.iloc[train_idx, 0].values
y_test = y.iloc[test_idx, 0].values

print(f"Training samples: {len(x_train)}, Test samples: {len(x_test)}")


dimension = x_train.shape
dimension_test = x_test.shape

print(x_train)
print(dimension)
print(y_test.shape)

In [ ]:
class_ratio = {}
class_numbering = {}

def calculate_class_ratio(y_train):
    for i in range(len(y_train)):
        if y_train[i] not in class_ratio:
            class_ratio[y_train[i]] = 1
        else:
            class_ratio[y_train[i]] += 1


#Assigns a number to different classes so can be used to compute probailities with y = k where k is a number
def calculate_class_numbering(y_train):
    count = 0
    for i in range(len(y_train)):
        if y_train[i] not in class_numbering:
            class_numbering[y_train[i]] = count
            count += 1

calculate_class_numbering(y_train)
calculate_class_ratio(y_train)

print(class_ratio)
print(class_numbering)


In [ ]:
#Will store the mean in the following format: (k, j) where k corresponds to the kth class and jth feature
mean_dict = {}

#Will store the variance in the following format: (k, j) where k corresponds to the kth class and jth feature
variance_dict = {}

def calculate_mean_dict(x_train, y_train):
    feature_number = 0
    for i in range(len(x_train[0])):
        #Using an auxilary dict to store means of a specific feature per class
        auxilary_dict = {}
        for j in range(len(x_train)):
            if y_train[j] not in auxilary_dict:
                auxilary_dict[y_train[j]] = (x_train[j][i]) / class_ratio[y_train[j]]
            else:
                auxilary_dict[y_train[j]] += x_train[j][i] / class_ratio[y_train[j]]
        
        for element in auxilary_dict:
            mean_dict[(element, feature_number)] = auxilary_dict[element]
        
        feature_number += 1

def calculate_variance_dict(x_train, y_train):
    feature_number = 0
    for i in range(len(x_train[0])):
        #Using an auxilary dict to store variance of a specific feature per class
        auxilary_dict = {}
        for j in range(len(x_train)):
            if y_train[j] not in auxilary_dict:
                auxilary_dict[y_train[j]] =  ((x_train[j][i]) ** 2) / class_ratio[y_train[j]]
            else:
                auxilary_dict[y_train[j]] += ((x_train[j][i]) ** 2) / class_ratio[y_train[j]]
        
        for element in auxilary_dict:
            variance_dict[(element, feature_number)] = auxilary_dict[element] - mean_dict[(element, feature_number)] ** 2
        
        feature_number += 1

calculate_mean_dict(x_train, y_train)
calculate_variance_dict(x_train, y_train)

# After calculate_mean_dict and calculate_variance_dict
print("Sample means:")
for key in list(mean_dict.keys())[:4]:
    print(f"  {key}: {mean_dict[key]}")

print("\nSample variances:")
for key in list(variance_dict.keys())[:4]:
    print(f"  {key}: {variance_dict[key]}")

print("\nClass ratios:")
print(class_ratio)

In [ ]:
#This function returns the probability of a specific feature. Will be used in predictions
def gaussian_probaulity(x, mean, variance):
    variance = max(variance, 1e-10)
    return max((1/math.sqrt(2 * math.pi * variance)) * math.exp(-((x - mean)**2) / (2 * variance)), 1e-300)

def predict_single(x):
    prediction = ''
    highest_probability = float('-inf')
    for class_type in class_numbering:
        probability_sum = math.log(class_ratio[class_type] / len(y_train)) 
        for j in range(len(x)):
            probability_sum += math.log(gaussian_probaulity(x[j], mean_dict[(class_type, j)], variance_dict[(class_type, j)]))
        

        if(probability_sum > highest_probability):
            prediction = class_type
            highest_probability = probability_sum
    
    return prediction

def predict_all(x_test):
    pred_vs_actual = []
    accuracy = 0
    for i in range(len(x_test)):
        prediction = predict_single(x_test[i])
        pred_vs_actual.append((prediction, y_test[i]))
        if(prediction == y_test[i]):
            accuracy += (1/len(x_test))
    
    return pred_vs_actual, accuracy

predicion_set, accuracy = predict_all(x_test)

print(predicion_set)
print(accuracy)